<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
پنهان‌کاری
</font>
</h1>

In [1]:
# import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مقدمه
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این تمرین قصد داریم تا دو تصویر را با همدیگر ترکیب کنیم. یک تصویر پس‌زمینه است و تصویر دیگر یک پیام است. ما قصد داریم تا این پیام را در تصویر پنهان کنیم. در ادامه مرحله به مرحله پیش خواهیم رفت تا در نهایت بتوانیم این متن را در تصویر جانشانی کنیم.
</font>
</p>

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله اول: وارد کردن تصاویر
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
ابتدا تصاویری که در پوشه Data قرار دارند را به صورت خاکستری وارد کنید.
<br>
این تصویر پیش‌زمینه است:
<br>
cover.png
<br>
این تصویری نیز متنی است که قرار است در نهایت در تصویر پنهان شود:
<br>
msg.png
</font>
</p>

In [2]:
"""
cover = cv2.imread('./Data/cover.png', cv2.IMREAD_GRAYSCALE)
msg = cv2.imread('./Data/msg.png', cv2.IMREAD_GRAYSCALE)
"""

"\ncover = cv2.imread('./Data/cover.png', cv2.IMREAD_GRAYSCALE)\nmsg = cv2.imread('./Data/msg.png', cv2.IMREAD_GRAYSCALE)\n"

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله دوم: اعمال فیلتر بالاگذر بر پیام
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این قسمت قرار است که یک فیلتر بالاگذر گاوسی را بر تصویر پیغام اعمال کنیم.  دقت کنید که فرکانس قطع را بدرستی انتخاب کنید. ممکن است پس از مراحل پایانی برای افزایش کیفیت به این مرحله بازگردید و فیلتر را با فرکانس قطع متفاوتی مجددا اعمال کنید.
<br>
پس از اعمال فیلتربالاگذر گاوسی در این مرحله باید متن‌ در تصویر واضح شود(همانند اینکه متن لبه‌ای در تصویر است) . 
</font>
<br>
پیشنهاد می‌شود که از آستانه‌های پیشنهادی زیر برای این فیلتر بالاگذر استفاده کنید و از بین آن‌ها بهترین آستانه را انتخاب کنید:
</p>

$$\text{thresholds\_high} = [10, 25, 50, 100, 150, 200, 500]$$

In [3]:
"""
thresholds_high = [150]
#[10, 25, 50, 100, 150, 200, 500]

for D0 in thresholds_high:
    dft = cv2.dft(np.float32(msg), flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)  # Shift DC component to center

    # Create Gaussian High-Pass Filter mask
    rows, cols = msg.shape
    crow, ccol = rows//2, cols//2  # Center coordinates

    u, v = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)  # Grid
    D = np.sqrt(u**2 + v**2)  # Distance from center
    H = 1 - np.exp(-(D**2)/(2*(D0**2)))  # Gaussian HPF formula
    H = cv2.merge([H, H])  # Convert to 2-channel format

    # Apply filter in frequency domain
    filtered_msg_dft = dft_shift * H
    msg_back = cv2.idft(np.fft.ifftshift(filtered_msg_dft), flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
    msg_back = cv2.normalize(msg_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

   
    # --- MSE ---
    mse = np.mean((msg_back.astype(np.float32) - msg_back.astype(np.float32))**2)
    ssim_val = ssim(msg, msg_back)
    print(f"D0 = {D0:3d} | MSE = {mse:.2f} | SSIM = {ssim_val:.4f}")
    
   
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Original Message Image')
plt.imshow(msg, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f'Filtered Message Image (D0={D0})')
plt.imshow(msg_back, cmap='gray')
plt.axis('off')
plt.show()

"""


'\nthresholds_high = [150]\n#[10, 25, 50, 100, 150, 200, 500]\n\nfor D0 in thresholds_high:\n    dft = cv2.dft(np.float32(msg), flags=cv2.DFT_COMPLEX_OUTPUT)\n    dft_shift = np.fft.fftshift(dft)  # Shift DC component to center\n\n    # Create Gaussian High-Pass Filter mask\n    rows, cols = msg.shape\n    crow, ccol = rows//2, cols//2  # Center coordinates\n\n    u, v = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)  # Grid\n    D = np.sqrt(u**2 + v**2)  # Distance from center\n    H = 1 - np.exp(-(D**2)/(2*(D0**2)))  # Gaussian HPF formula\n    H = cv2.merge([H, H])  # Convert to 2-channel format\n\n    # Apply filter in frequency domain\n    filtered_msg_dft = dft_shift * H\n    msg_back = cv2.idft(np.fft.ifftshift(filtered_msg_dft), flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)\n    msg_back = cv2.normalize(msg_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)\n\n   \n    # --- MSE ---\n    mse = np.mean((msg_back.astype(np.float32) - msg_back.astype(np.float32))**2)\n   

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله سوم: اعمال فیلتر پایین‌گذر بر پیش‌زمینه
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این قسمت قرار است که یک فیلتر پایین‌گذر گاوسی را بر تصویر پیش‌زمینه اعمال کنیم.  دقت کنید که فرکانس قطع را بدرستی انتخاب کنید. ممکن است پس از مراحل پایانی برای افزایش کیفیت به این مرحله بازگردید و فیلتر را با فرکانس قطع متفاوتی مجددا اعمال کنید.
<br>
این فیلتر جزئیات نرم و کلی تصویر را حفظ می‌کند، در حالی که جزئیات با فرکانس بالا را کاهش می‌دهد.
</font>
<br>
پیشنهاد می‌شود که از آستانه‌های پیشنهادی زیر برای این فیلتر پاین‌گذر استفاده کنید و از بین آن‌ها بهترین آستانه را انتخاب کنید:
</p>

$$\text{thresholds\_low} = [10, 50, 100, 250, 500]$$

In [4]:
"""
thresholds_low = [250]
#[10, 50, 100, 250, 500]

for D0 in thresholds_low:
    dft = cv2.dft(np.float32(cover), flags=cv2.DFT_COMPLEX_OUTPUT)
    dft_shift = np.fft.fftshift(dft)  # Shift DC component to center

    # Create Gaussian High-Pass Filter mask
    rows, cols = cover.shape
    crow, ccol = rows//2, cols//2  # Center coordinates

    u, v = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)  # Grid
    D = np.sqrt(u**2 + v**2)  # Distance from center
    H = np.exp(-(D**2)/(2*(D0**2)))  # Gaussian HPF formula
    H = cv2.merge([H, H])  # Convert to 2-channel format

    # Apply filter in frequency domain
    filtered_cover_dft = dft_shift * H
    cover_back = cv2.idft(np.fft.ifftshift(filtered_cover_dft), flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
    cover_back = cv2.normalize(cover_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    
    # --- MSE ---
    mse = np.mean((cover.astype(np.float32) - cover_back.astype(np.float32))**2)

    # --- SSIM ---
    ssim_val = ssim(cover, cover_back)
    
    print(f"D0 = {D0:3d} | MSE = {mse:.2f} | SSIM = {ssim_val:.4f}")

    

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Original Cover Image')
plt.imshow(cover, cmap='gray')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f'Filtered Cover Image (D0={D0})')
plt.imshow(cover_back, cmap='gray')
plt.axis('off')
plt.show()

"""

'\nthresholds_low = [250]\n#[10, 50, 100, 250, 500]\n\nfor D0 in thresholds_low:\n    dft = cv2.dft(np.float32(cover), flags=cv2.DFT_COMPLEX_OUTPUT)\n    dft_shift = np.fft.fftshift(dft)  # Shift DC component to center\n\n    # Create Gaussian High-Pass Filter mask\n    rows, cols = cover.shape\n    crow, ccol = rows//2, cols//2  # Center coordinates\n\n    u, v = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)  # Grid\n    D = np.sqrt(u**2 + v**2)  # Distance from center\n    H = np.exp(-(D**2)/(2*(D0**2)))  # Gaussian HPF formula\n    H = cv2.merge([H, H])  # Convert to 2-channel format\n\n    # Apply filter in frequency domain\n    filtered_cover_dft = dft_shift * H\n    cover_back = cv2.idft(np.fft.ifftshift(filtered_cover_dft), flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)\n    cover_back = cv2.normalize(cover_back, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)\n\n    \n    # --- MSE ---\n    mse = np.mean((cover.astype(np.float32) - cover_back.astype(np.float32))**2)\n\n  

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله چهارم: ترکیب طیف‌های فرکانسی
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
در این مرحله نوبت به آن می‌رسد که طیف‌های فرکانسی  که در مراحل قبلی محاسبه کرده بودیم را با یکدیگر ترکیب کنیم. 
<br>
ابتدا هر دو طیف فرکانسی را با یکدیگر جمع کنید. 
<br>
پس از جمع هر دو طیف فرکانسی، طیف فرکانسی تصویری بدست می‌آید که معادل است با طیف فرکانسی تصویری که متن و پیش‌زمینه با هم ترکیب شده‌اند. باید این طیف فرکانسی نهایی را از حوزه فرکانس به حوزه مکانی برگردانید.
</p>

In [5]:
"""

watermarked_dft = filtered_cover_dft + filtered_msg_dft
watermarked_dft_shift = np.fft.ifftshift(watermarked_dft)

watermarked = cv2.idft(
    watermarked_dft_shift,
    flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT
)

watermarked = watermarked.astype(np.uint8)

plt.imshow(watermarked, cmap='gray')
plt.axis('off')
plt.show()

"""


"\n\nwatermarked_dft = filtered_cover_dft + filtered_msg_dft\nwatermarked_dft_shift = np.fft.ifftshift(watermarked_dft)\n\nwatermarked = cv2.idft(\n    watermarked_dft_shift,\n    flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT\n)\n\nwatermarked = watermarked.astype(np.uint8)\n\nplt.imshow(watermarked, cmap='gray')\nplt.axis('off')\nplt.show()\n\n"

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله پنجم: ذخیره‌سازی
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
حال در این مرحله، اگر تصویر ترکیب شده دارای کیفیت مناسب بود، این  تصویر نهایی را در کنار این نوت‌بوک ذخیره کنید. دقت کنید که حتما تصویر را در حوزه خاکستری ذخیره کنید. نام تصویر نهایی عبارت است از:
<br>
final_result.png
</p>

In [6]:
cover = cv2.imread('./Data/cover.png', cv2.IMREAD_GRAYSCALE)
msg   = cv2.imread('./Data/msg.png',   cv2.IMREAD_GRAYSCALE)

# اگر ابعاد یکی نیست (خیلی مهم)
if msg.shape != cover.shape:
    msg = cv2.resize(msg, (cover.shape[1], cover.shape[0]))

D0_high = 150   # پیام
D0_low  = 250   # کاور

rows, cols = cover.shape
crow, ccol = rows//2, cols//2
u, v = np.meshgrid(np.arange(cols)-ccol, np.arange(rows)-crow)
D = np.sqrt(u**2 + v**2)

# ماسک‌ها (تک‌کاناله)
H_hp = 1 - np.exp(-(D**2)/(2*(D0_high**2)))   # high-pass
H_lp =     np.exp(-(D**2)/(2*(D0_low**2)))    # low-pass

# FFT
F_msg   = np.fft.fftshift(np.fft.fft2(msg.astype(np.float32)))
F_cover = np.fft.fftshift(np.fft.fft2(cover.astype(np.float32)))

# فیلتر در حوزه فرکانس
F_msg_hp   = F_msg   * H_hp
F_cover_lp = F_cover * H_lp

# ترکیب طیف‌ها
F_watermarked = F_cover_lp + F_msg_hp

# بازگشت به حوزه مکانی (نکته مربی: np.real)
watermarked = np.real(np.fft.ifft2(np.fft.ifftshift(F_watermarked)))

# تبدیل به تصویر 0..255 (برای ذخیره نهایی)
watermarked = cv2.normalize(watermarked, None, 0, 255, cv2.NORM_MINMAX)
watermarked = watermarked.astype(np.uint8)

cv2.imwrite('final_result.png', watermarked)


True

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مرحله پایانی: سلول جواب‌ساز
</font>
</h3>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
بدون هیچگونه تغییری، این سلول را اجرا کنید تا فایل پاسخ شما آماده شود.
</font>
</p>

In [7]:
import zipfile

def compress(file_names):
    print("File Paths:")
    print(file_names)
    # Select the compression mode ZIP_DEFLATED for compression
    # or zipfile.ZIP_STORED to just store the file
    compression = zipfile.ZIP_DEFLATED
    # create the zip file first parameter path/name, second mode
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            # Add file to the zip file
            # first parameter file to zip, second filename in zip
            zf.write('./' + file_name, file_name, compress_type=compression)


file_names = ["notebook.ipynb", "final_result.png"]
compress(file_names)

File Paths:
['notebook.ipynb', 'final_result.png']
